# Rock Fragment Analysis — SAM 2 Zero-Shot Pipeline

## What this notebook does
1. **Scale calibration** — detects the red-tipped survey rod in the image and computes a pixel → mm ratio  
2. **Instance segmentation** — uses Meta's SAM 2 to generate individual rock masks with no training required  
3. **Fragment measurement** — computes real physical dimensions (ECD, axes, aspect ratio) for every detected fragment  
4. **Size distribution** — fits a Rosin-Rammler curve and reports P20/P50/P80 — the standard mining KPIs  
5. **Visualisation** — produces a color-coded fragment map overlaid on the original image  

---

## Before you run anything — read this

### Runtime requirement
- **Google Colab**: Go to `Runtime → Change runtime type → T4 GPU`. SAM 2 will not run in reasonable time on CPU only.
- **Local Jupyter**: You need an NVIDIA GPU with CUDA 11.8+ or 12.x and at least 6 GB VRAM. Run `nvidia-smi` in a terminal to confirm.

### How to use your own image
- In Colab: click the folder icon on the left sidebar → upload your image → set `IMAGE_PATH` in Cell 4 to the filename.
- The image must have the survey rod (scale stick) fully visible for calibration to work.
- If the rod is not visible, set `ROD_KNOWN_LENGTH_MM = None` and the pipeline will skip calibration and report sizes in pixels.

### Survey rod configuration (Cell 4)
- `ROD_KNOWN_LENGTH_MM`: real-world length of the rod in mm. Standard surveying rods are 2000 mm. Measure yours if unsure.
- `ROD_COLOR`: `'red'` for red-tipped rods (default). Change to `'yellow'` or `'orange'` if your rod is a different colour.

---

## Cell 1 — Install packages

**Run this cell first, then wait for it to finish completely.**  
It installs SAM 2 from Meta's official GitHub repository and all supporting libraries.  
Expected time: ~2–4 minutes on Colab. The cell will print many lines — that is normal.  
After it finishes, **do not restart the runtime** unless you see an error.

In [ ]:
# ── Install SAM 2 and all dependencies ──────────────────────────────────────
# sam2 is installed from Meta's GitHub because the PyPI package (sam-2) is
# often behind the latest checkpoint format. This takes 2–4 minutes.

!pip install -q git+https://github.com/facebookresearch/sam2.git
!pip install -q opencv-python-headless scikit-image pandas matplotlib scipy seaborn tqdm

print("\n✅ All packages installed.")

## Cell 2 — Download SAM 2 model checkpoint

**Run after Cell 1 completes.**  

This downloads the SAM 2.1 Large checkpoint (~900 MB) from Meta's servers.  
The file is saved to `./checkpoints/` in your Colab working directory.  

**Checkpoint options** (change `CHECKPOINT` variable below if needed):  
| Checkpoint | Size | Speed | Quality |
|---|---|---|---|
| `sam2.1_hiera_large.pt` | 900 MB | Slowest | Best ← default |
| `sam2.1_hiera_base_plus.pt` | 320 MB | Medium | Good |
| `sam2.1_hiera_small.pt` | 185 MB | Fast | Fair |
| `sam2.1_hiera_tiny.pt` | 155 MB | Fastest | Weakest |

For rock fragment analysis with irregular shapes, use `large`. Only switch to smaller checkpoints if you hit VRAM limits.

In [ ]:
import os

# ── Choose checkpoint ────────────────────────────────────────────────────────
CHECKPOINT = "sam2.1_hiera_large.pt"       # change to _base_plus / _small / _tiny if needed
CONFIG     = "configs/sam2.1/sam2.1_hiera_l.yaml"  # must match checkpoint

# Config name lookup — update this dict if you change CHECKPOINT
_CONFIG_MAP = {
    "sam2.1_hiera_large.pt":     "configs/sam2.1/sam2.1_hiera_l.yaml",
    "sam2.1_hiera_base_plus.pt": "configs/sam2.1/sam2.1_hiera_b+.yaml",
    "sam2.1_hiera_small.pt":     "configs/sam2.1/sam2.1_hiera_s.yaml",
    "sam2.1_hiera_tiny.pt":      "configs/sam2.1/sam2.1_hiera_t.yaml",
}
CONFIG = _CONFIG_MAP[CHECKPOINT]

os.makedirs("checkpoints", exist_ok=True)
ckpt_path = os.path.join("checkpoints", CHECKPOINT)

if os.path.exists(ckpt_path):
    print(f"✅ Checkpoint already downloaded: {ckpt_path}")
else:
    base_url = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/"
    print(f"⬇  Downloading {CHECKPOINT} (~900 MB for large) ...")
    !wget -q --show-progress -O {ckpt_path} {base_url}{CHECKPOINT}
    print(f"\n✅ Checkpoint saved to {ckpt_path}")

## Cell 3 — Imports

**Run after the checkpoint download completes.**  
No user action needed here — just run the cell.  
If you see `ModuleNotFoundError: sam2`, go back and re-run Cell 1.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import warnings
from pathlib import Path
from tqdm import tqdm
from scipy.optimize import curve_fit
from scipy import stats
from skimage import measure
import torch

from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Imports OK  |  Device: {device}")
if device == "cpu":
    print("⚠️  No GPU detected. SAM 2 will be very slow (10–30 min per image).")
    print("   In Colab: Runtime → Change runtime type → T4 GPU, then re-run all cells.")

## Cell 4 — Configuration

**This is the only cell you need to edit for each new image.**  

- Set `IMAGE_PATH` to the path of your uploaded image.
- Set `ROD_KNOWN_LENGTH_MM` to the real-world length of your survey rod in millimetres. A standard 2 m rod = 2000 mm.
- Set `ROD_COLOR` to match your rod tip colour (`'red'`, `'orange'`, or `'yellow'`).
- `MIN_FRAGMENT_AREA_PX` filters out dust and debris smaller than this many pixels. Increase if you're getting too many micro-fragments.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  USER CONFIGURATION — edit these values for each image
# ═══════════════════════════════════════════════════════════════════════════

IMAGE_PATH             = "shared_image__29_.jpg"  # ← path to your uploaded image

# Scale calibration
ROD_KNOWN_LENGTH_MM    = 2000          # real-world rod length in mm (2 m rod = 2000)
                                       # Set to None to skip calibration (sizes in pixels)
ROD_COLOR              = "red"         # tip colour: 'red', 'orange', or 'yellow'

# Fragment filtering
MIN_FRAGMENT_AREA_PX   = 800           # ignore masks smaller than this (filters dust/noise)
MAX_FRAGMENT_AREA_FRAC = 0.15          # ignore masks covering more than 15% of image (sky/ground)

# SAM 2 mask generator sensitivity
# Lower points_per_side = faster but misses small fragments
# Higher = slower but more complete. 32 is a good default for rock piles.
POINTS_PER_SIDE        = 32
PRED_IOU_THRESH        = 0.86          # confidence threshold — lower to detect more fragments
STABILITY_SCORE_THRESH = 0.92          # mask stability — lower if rocks are poorly lit

# ═══════════════════════════════════════════════════════════════════════════

print(f"Image path      : {IMAGE_PATH}")
print(f"Rod length      : {ROD_KNOWN_LENGTH_MM} mm")
print(f"Rod colour      : {ROD_COLOR}")
print(f"Min area filter : {MIN_FRAGMENT_AREA_PX} px")
print(f"SAM sensitivity : {POINTS_PER_SIDE} pts/side, IOU≥{PRED_IOU_THRESH}, stability≥{STABILITY_SCORE_THRESH}")

## Cell 5 — Scale calibration

**Run after setting configuration.**  

This cell detects the survey rod tips using HSV colour thresholding, then computes the pixel-to-mm ratio.  
It will display the image with detected tip positions marked so you can verify the detection visually.

**If calibration fails** (tips not found):  
- Check that the rod is clearly visible and the tips are not obscured by shadow or rock  
- Try adjusting `ROD_COLOR` or `_HSV_RANGES` inside this cell  
- As a fallback, you can manually set `pixel_to_mm` at the bottom of this cell (e.g. `pixel_to_mm = 0.35`)  
  Tip: if you know the rod appears N pixels long in the image, `pixel_to_mm = ROD_KNOWN_LENGTH_MM / N`

In [ ]:
# ── HSV ranges for tip colour detection ─────────────────────────────────────
# Each entry is (lower_hsv, upper_hsv). Red wraps around in HSV so two ranges.
_HSV_RANGES = {
    "red":    [(np.array([0,  120, 70]),  np.array([10, 255, 255])),
               (np.array([170,120, 70]),  np.array([180,255, 255]))],
    "orange": [(np.array([8,  150, 80]),  np.array([25, 255, 255]))],
    "yellow": [(np.array([22, 120, 80]),  np.array([35, 255, 255]))],
}

def _detect_rod_tips(image_bgr, color):
    """Return (tip1_px, tip2_px) as (x,y) pixel coords, or None if not found."""
    hsv   = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    mask  = np.zeros(hsv.shape[:2], dtype=np.uint8)
    for (lo, hi) in _HSV_RANGES.get(color, _HSV_RANGES["red"]):
        mask = cv2.bitwise_or(mask, cv2.inRange(hsv, lo, hi))

    # Clean up noise
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel, iterations=2)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_DILATE, kernel, iterations=1)

    # Find connected components — keep only the two largest blobs
    n_labels, labels, stats_cc, centroids = cv2.connectedComponentsWithStats(mask)
    if n_labels < 3:          # 3 because label 0 is background
        return None, mask

    areas     = stats_cc[1:, cv2.CC_STAT_AREA]   # skip background
    top2_idx  = np.argsort(areas)[-2:] + 1        # +1 to re-align with labels
    tip_coords = [tuple(centroids[i].astype(int)) for i in top2_idx]
    return tip_coords, mask


# ── Load image ───────────────────────────────────────────────────────────────
image_bgr  = cv2.imread(IMAGE_PATH)
if image_bgr is None:
    raise FileNotFoundError(
        f"Could not load '{IMAGE_PATH}'.\n"
        "In Colab: upload the image via the folder icon on the left sidebar first."
    )
image_rgb  = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
img_h, img_w = image_rgb.shape[:2]
print(f"Image loaded: {img_w} × {img_h} px")

# ── Calibrate ────────────────────────────────────────────────────────────────
pixel_to_mm = None   # will be set below

if ROD_KNOWN_LENGTH_MM is not None:
    tips, tip_mask = _detect_rod_tips(image_bgr, ROD_COLOR)

    if tips is not None:
        (x1, y1), (x2, y2) = tips
        rod_length_px = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
        pixel_to_mm   = ROD_KNOWN_LENGTH_MM / rod_length_px

        print(f"✅ Rod detected    : tip1=({x1},{y1})  tip2=({x2},{y2})")
        print(f"   Rod length px   : {rod_length_px:.1f} px")
        print(f"   Pixel → mm ratio: {pixel_to_mm:.4f} mm/px  ({1/pixel_to_mm:.1f} px/mm)")

        # ── Visualise detection ───────────────────────────────────────────────
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        overlay = image_rgb.copy()
        cv2.circle(overlay, (x1, y1), 18, (255, 255,   0), 4)
        cv2.circle(overlay, (x2, y2), 18, (255, 255,   0), 4)
        cv2.line(overlay,   (x1, y1), (x2, y2), (255, 255, 0), 3)
        label_pt = ((x1+x2)//2 + 20, (y1+y2)//2 - 20)
        cv2.putText(overlay, f"{ROD_KNOWN_LENGTH_MM} mm", label_pt,
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 0), 3)

        axes[0].imshow(overlay)
        axes[0].set_title("Detected survey rod (yellow line)", fontweight="bold")
        axes[0].axis("off")
        axes[1].imshow(tip_mask, cmap="hot")
        axes[1].set_title(f"HSV colour mask ({ROD_COLOR} tips)", fontweight="bold")
        axes[1].axis("off")
        plt.tight_layout()
        plt.show()

    else:
        print("⚠️  Rod tips not detected automatically.")
        print("   Options:")
        print("   1) Manually set pixel_to_mm below (uncomment and set value)")
        print("   2) Change ROD_COLOR in Cell 4 and re-run")
        print("   3) Set ROD_KNOWN_LENGTH_MM = None in Cell 4 to skip calibration")
        # ── Manual override (uncomment if auto-detection fails) ───────────────
        # pixel_to_mm = 0.35   # ← set this to ROD_KNOWN_LENGTH_MM / rod_pixels

if pixel_to_mm is None:
    pixel_to_mm = 1.0
    print("ℹ️  Calibration skipped — sizes will be reported in pixels, not mm.")
    print("   Set ROD_KNOWN_LENGTH_MM and ensure rod is visible for real-world sizing.")

## Cell 6 — Load SAM 2 model

**Run once per session.**  
This loads the checkpoint into GPU memory (~2–3 GB for the large model).  
Expected time: ~30–60 seconds.  

If you see `CUDA out of memory`, either switch to the `_base_plus` checkpoint in Cell 2/4 or reduce your Colab runtime to avoid other memory users.

In [ ]:
print(f"Loading SAM 2 ({CHECKPOINT}) on {device} ...")

sam2_model = build_sam2(
    config_file   = CONFIG,
    ckpt_path     = ckpt_path,
    device        = device,
    apply_postprocessing = False,   # keep masks sharp, avoid boundary smoothing
)

mask_generator = SAM2AutomaticMaskGenerator(
    model                  = sam2_model,
    points_per_side        = POINTS_PER_SIDE,
    pred_iou_thresh        = PRED_IOU_THRESH,
    stability_score_thresh = STABILITY_SCORE_THRESH,
    box_nms_thresh         = 0.7,          # suppress heavily overlapping duplicate masks
    min_mask_region_area   = MIN_FRAGMENT_AREA_PX,
    output_mode            = "binary_mask",
)

print(f"✅ SAM 2 loaded and ready.")
if device == "cuda":
    used_gb = torch.cuda.memory_allocated() / 1e9
    print(f"   GPU memory used: {used_gb:.1f} GB")

## Cell 7 — Run SAM 2 segmentation

**This is the main inference step.**  
SAM 2 samples a grid of points across the image and generates a mask for every distinct region it finds.  

Expected time: ~30–90 seconds on a T4 GPU depending on image size and `POINTS_PER_SIDE`.  

**Understanding the output:**  
Each mask in `all_masks` is a dict with:
- `segmentation` — boolean 2D array (True = fragment pixel)
- `area` — pixel count
- `predicted_iou` — model's confidence in this mask (0–1)
- `bbox` — bounding box [x, y, w, h]

After filtering, `fragment_masks` contains only masks in the valid size range.

In [ ]:
print("Running SAM 2 automatic mask generation ...")
print("(This may take 30–90 seconds on GPU)")

# SAM 2 expects uint8 RGB
with torch.inference_mode():
    all_masks = mask_generator.generate(image_rgb)

print(f"SAM 2 found {len(all_masks)} raw masks")

# ── Filter masks by area ─────────────────────────────────────────────────────
img_area          = img_h * img_w
max_area_px       = MAX_FRAGMENT_AREA_FRAC * img_area

fragment_masks = [
    m for m in all_masks
    if MIN_FRAGMENT_AREA_PX <= m["area"] <= max_area_px
]

# Sort by area descending (largest fragments first)
fragment_masks.sort(key=lambda m: m["area"], reverse=True)

print(f"\nAfter size filtering ({MIN_FRAGMENT_AREA_PX}–{int(max_area_px)} px):")
print(f"  ✅ {len(fragment_masks)} fragment masks retained")
print(f"  ✗  {len(all_masks) - len(fragment_masks)} masks removed (too small or too large)")

# ── Quick preview — first 12 masks ──────────────────────────────────────────
n_preview = min(12, len(fragment_masks))
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
fig.suptitle(f"SAM 2 mask preview — top {n_preview} largest fragments",
             fontsize=14, fontweight="bold")
for i, ax in enumerate(axes.flat):
    if i < n_preview:
        m    = fragment_masks[i]
        crop = image_rgb.copy()
        crop[~m["segmentation"]] = (crop[~m["segmentation"]] * 0.35).astype(np.uint8)
        x, y, w, h = m["bbox"]
        pad  = 20
        x0, y0 = max(0, x-pad), max(0, y-pad)
        x1, y1 = min(img_w, x+w+pad), min(img_h, y+h+pad)
        ax.imshow(crop[y0:y1, x0:x1])
        ax.set_title(f"Frag {i+1}  IOU={m['predicted_iou']:.2f}  {m['area']} px",
                     fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Cell 8 — Measure fragments

**Run after segmentation completes.**  
No configuration needed.  

This cell computes physical measurements for each fragment mask using `skimage.measure.regionprops`.  

**Metrics computed per fragment:**
- **ECD** (Equivalent Circular Diameter): `√(4·area/π)` in mm — the industry standard size metric. Equivalent to the sieve size the fragment would pass through.
- **Major / minor axis**: longest and shortest span of the fragment in mm
- **Aspect ratio**: major/minor — values close to 1 = blocky, large values = flaky/elongated
- **Solidity**: area / convex hull area — how convex/jagged the fragment is (1 = perfect convex shape)

In [ ]:
records = []

for i, m in enumerate(tqdm(fragment_masks, desc="Measuring fragments")):
    seg  = m["segmentation"].astype(np.uint8)
    props_list = measure.regionprops(seg)
    if not props_list:
        continue
    p = props_list[0]

    area_px       = p.area
    major_px      = p.major_axis_length
    minor_px      = p.minor_axis_length if p.minor_axis_length > 0 else 1.0
    ecd_px        = np.sqrt(4 * area_px / np.pi)     # equivalent circular diameter in px

    # Convert to mm
    ecd_mm        = ecd_px   * pixel_to_mm
    major_mm      = major_px * pixel_to_mm
    minor_mm      = minor_px * pixel_to_mm

    aspect_ratio  = major_px / minor_px
    solidity      = p.solidity                        # area / convex_hull_area
    centroid_x, centroid_y = p.centroid               # (row, col)

    records.append({
        "fragment_id":    i + 1,
        "area_px":        area_px,
        "ecd_mm":         round(ecd_mm, 2),
        "major_axis_mm":  round(major_mm, 2),
        "minor_axis_mm":  round(minor_mm, 2),
        "aspect_ratio":   round(aspect_ratio, 3),
        "solidity":       round(solidity, 3),
        "iou_score":      round(m["predicted_iou"], 3),
        "centroid_x_px":  round(centroid_y),          # note: centroid=(row,col) so swap
        "centroid_y_px":  round(centroid_x),
    })

df = pd.DataFrame(records)

unit = "mm" if pixel_to_mm != 1.0 else "px"
print(f"\n{'='*55}")
print(f"  Fragment measurement summary  ({unit})")
print(f"{'='*55}")
print(f"  Total fragments measured : {len(df)}")
print(f"  ECD  — mean   : {df.ecd_mm.mean():.1f} {unit}")
print(f"  ECD  — median : {df.ecd_mm.median():.1f} {unit}")
print(f"  ECD  — P20    : {df.ecd_mm.quantile(0.20):.1f} {unit}")
print(f"  ECD  — P50    : {df.ecd_mm.quantile(0.50):.1f} {unit}")
print(f"  ECD  — P80    : {df.ecd_mm.quantile(0.80):.1f} {unit}")
print(f"  ECD  — max    : {df.ecd_mm.max():.1f} {unit}")
print(f"  Aspect ratio  : mean={df.aspect_ratio.mean():.2f}")
print(f"  Solidity      : mean={df.solidity.mean():.2f}")
print(f"{'='*55}")
df.head(10)

## Cell 9 — Rosin-Rammler distribution fit

**Run after Cell 8.**  

The Rosin-Rammler (Weibull) distribution is the industry standard for blasting fragmentation.  
It describes cumulative percent passing vs sieve size:  

$$P(x) = 1 - e^{-(x / x_c)^n}$$

Where:  
- **xc** (characteristic size): the sieve size at which 63.2% of material passes  
- **n** (uniformity index): shape of the curve (higher = narrower size distribution = more uniform blasting)  

**Reading the P80 result:**  
P80 is the sieve size that 80% of the material passes through. This is the primary KPI used in blast design — it directly determines crusher feed requirements.

In [ ]:
def rosin_rammler(x, xc, n):
    """Cumulative percent passing: P(x) = 1 - exp(-(x/xc)^n)"""
    return 1.0 - np.exp(-((x / xc) ** n))

# Build cumulative distribution from measured ECDs
sizes_sorted    = np.sort(df["ecd_mm"].values)
cum_frac        = np.arange(1, len(sizes_sorted) + 1) / len(sizes_sorted)

# Fit Rosin-Rammler — initial guess: xc=median, n=1.5
try:
    popt, pcov = curve_fit(
        rosin_rammler, sizes_sorted, cum_frac,
        p0=[np.median(sizes_sorted), 1.5],
        bounds=([0.1, 0.1], [np.inf, 20]),
        maxfev=5000
    )
    xc_fit, n_fit = popt
    fit_ok = True
except RuntimeError:
    fit_ok = False
    print("⚠️  Rosin-Rammler fit did not converge. Plotting empirical CDF only.")

# Derive P-values from the fit
if fit_ok:
    def inv_rr(p, xc, n):
        """Inverse: size at cumulative fraction p"""
        return xc * (-np.log(1 - p)) ** (1 / n)

    p20 = inv_rr(0.20, xc_fit, n_fit)
    p50 = inv_rr(0.50, xc_fit, n_fit)
    p80 = inv_rr(0.80, xc_fit, n_fit)

unit = "mm" if pixel_to_mm != 1.0 else "px"

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: CDF
ax = axes[0]
ax.scatter(sizes_sorted, cum_frac * 100, s=4, alpha=0.4,
           color="steelblue", label="Measured fragments")
if fit_ok:
    x_fit = np.linspace(sizes_sorted.min(), sizes_sorted.max(), 500)
    ax.plot(x_fit, rosin_rammler(x_fit, xc_fit, n_fit) * 100,
            "r-", linewidth=2, label=f"Rosin-Rammler  xc={xc_fit:.1f} {unit}, n={n_fit:.2f}")
    for p_val, p_size, color in [(20, p20, "#2196F3"), (50, p50, "#FF9800"), (80, p80, "#F44336")]:
        ax.axhline(p_val, color=color, linestyle="--", linewidth=1, alpha=0.7)
        ax.axvline(p_size, color=color, linestyle="--", linewidth=1, alpha=0.7)
        ax.annotate(f"P{p_val}={p_size:.0f}{unit}",
                    xy=(p_size, p_val), xytext=(p_size + 2, p_val + 3),
                    fontsize=9, color=color, fontweight="bold")
ax.set_xlabel(f"Fragment size ECD ({unit})", fontsize=11)
ax.set_ylabel("Cumulative passing (%)", fontsize=11)
ax.set_title("Rosin-Rammler size distribution", fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Right: Histogram
ax = axes[1]
ax.hist(df["ecd_mm"], bins=40, color="steelblue", edgecolor="white",
        linewidth=0.5, alpha=0.85)
if fit_ok:
    ax.axvline(p80, color="#F44336", linestyle="--", linewidth=2,
               label=f"P80 = {p80:.0f} {unit}")
    ax.axvline(p50, color="#FF9800", linestyle="--", linewidth=2,
               label=f"P50 = {p50:.0f} {unit}")
    ax.legend(fontsize=10)
ax.set_xlabel(f"Fragment size ECD ({unit})", fontsize=11)
ax.set_ylabel("Count", fontsize=11)
ax.set_title("Fragment size histogram", fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

if fit_ok:
    print(f"\nRosin-Rammler fit parameters")
    print(f"  xc (characteristic size) : {xc_fit:.1f} {unit}")
    print(f"  n  (uniformity index)    : {n_fit:.3f}")
    print(f"  P20 = {p20:.1f} {unit}")
    print(f"  P50 = {p50:.1f} {unit}")
    print(f"  P80 = {p80:.1f} {unit}  ← primary blast KPI")

## Cell 10 — Color-coded fragment map

**Run after Cell 8 (measurement).**  
No configuration needed.  

This produces the main visual output: the original image with every detected fragment filled with a colour representing its size class.  
The six size classes follow standard blast fragmentation terminology.  

**Tip:** If you want to change the size class boundaries to match your specific project specs,  
edit the `SIZE_CLASSES` dict in this cell.

In [ ]:
# ── Size class definitions (edit boundaries here if needed) ──────────────────
# Keys are labels, values are (min_mm, max_mm, RGB_color)
SIZE_CLASSES = {
    "Very fine  < 20 mm":   (0,    20,   (255,  50, 255)),   # magenta
    "Fine  20–50 mm":       (20,   50,   (255,  50,  50)),   # red
    "Medium  50–100 mm":    (50,  100,   (255, 165,  30)),   # orange
    "Coarse  100–200 mm":   (100, 200,   (255, 230,  30)),   # yellow
    "Large  200–400 mm":    (200, 400,   ( 50, 200,  50)),   # green
    "Boulder  > 400 mm":    (400, 9999,  ( 50,  50, 255)),   # blue
}

def _size_to_color(ecd_mm):
    for label, (lo, hi, col) in SIZE_CLASSES.items():
        if lo <= ecd_mm < hi:
            return col, label
    return (128, 128, 128), "Unknown"

# Build a lookup: fragment_id → color, class
df["size_class"] = ""
df["color_rgb"]  = None
for idx, row in df.iterrows():
    col, cls = _size_to_color(row["ecd_mm"])
    df.at[idx, "color_rgb"]  = col
    df.at[idx, "size_class"] = cls

# ── Build the overlay image ───────────────────────────────────────────────────
overlay = image_rgb.astype(np.float32).copy()
alpha   = 0.55    # transparency: 0 = original only, 1 = full color

for i, m in enumerate(fragment_masks[:len(df)]):
    row   = df.iloc[i]
    color = row["color_rgb"]
    seg   = m["segmentation"]
    for c_idx, c_val in enumerate(color):
        overlay[seg, c_idx] = (
            alpha * c_val + (1 - alpha) * overlay[seg, c_idx]
        )

# Draw fragment boundaries
overlay_uint8 = np.clip(overlay, 0, 255).astype(np.uint8)
for m in fragment_masks[:len(df)]:
    contours, _ = cv2.findContours(
        m["segmentation"].astype(np.uint8),
        cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    cv2.drawContours(overlay_uint8, contours, -1, (255, 255, 255), 1)

# ── Main figure ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(22, 10))

axes[0].imshow(image_rgb)
axes[0].set_title("Original image", fontweight="bold", fontsize=13)
axes[0].axis("off")

axes[1].imshow(overlay_uint8)
axes[1].set_title("Fragment size map (SAM 2 instance segmentation)",
                  fontweight="bold", fontsize=13)
axes[1].axis("off")

# Legend
legend_patches = [
    mpatches.Patch(
        facecolor=tuple(c/255 for c in color),
        edgecolor="white", linewidth=0.5, label=label
    )
    for label, (_, _, color) in SIZE_CLASSES.items()
]
axes[1].legend(
    handles=legend_patches, loc="lower right",
    fontsize=10, framealpha=0.85,
    title="Fragment size (ECD)", title_fontsize=10
)

plt.tight_layout()
plt.show()

# ── Size class distribution bar chart ────────────────────────────────────────
class_counts = df["size_class"].value_counts().reindex(
    [lbl for lbl in SIZE_CLASSES.keys()], fill_value=0
)
class_pcts   = class_counts / class_counts.sum() * 100
bar_colors   = [tuple(c/255 for c in v[2]) for v in SIZE_CLASSES.values()]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(range(len(class_counts)), class_pcts.values,
              color=bar_colors, edgecolor="white", linewidth=0.8)
ax.set_xticks(range(len(class_counts)))
ax.set_xticklabels(class_counts.index, rotation=20, ha="right", fontsize=10)
ax.set_ylabel("Proportion of fragments (%)", fontsize=11)
ax.set_title("Fragment size class distribution", fontweight="bold", fontsize=12)
ax.grid(True, alpha=0.3, axis="y")
for bar, pct in zip(bars, class_pcts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
            f"{pct:.1f}%", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

print("\nSize class breakdown:")
for label, count, pct in zip(class_counts.index, class_counts.values, class_pcts.values):
    print(f"  {label:<25} {count:>5} fragments  ({pct:.1f}%)")

## Cell 11 — Export results to CSV

**Optional — run to save measurements.**  

Saves the full per-fragment measurement table to a CSV file.  
In Colab, right-click the file in the left sidebar → Download to get it to your local machine.

In [ ]:
output_csv = Path(IMAGE_PATH).stem + "_fragment_measurements.csv"
df_export  = df.drop(columns=["color_rgb"]).copy()   # color tuples don't serialise well

# Add derived unit column
unit = "mm" if pixel_to_mm != 1.0 else "px"
df_export.rename(columns={
    "ecd_mm":        f"ecd_{unit}",
    "major_axis_mm": f"major_axis_{unit}",
    "minor_axis_mm": f"minor_axis_{unit}",
}, inplace=True)

df_export.to_csv(output_csv, index=False)
print(f"✅ Saved {len(df_export)} fragment records → {output_csv}")
df_export.describe().round(2)

---

## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `ModuleNotFoundError: sam2` | Cell 1 didn't complete | Re-run Cell 1; check for errors in the output |
| `FileNotFoundError` on IMAGE_PATH | Image not uploaded or wrong name | In Colab: upload via folder icon; verify exact filename in Cell 4 |
| `CUDA out of memory` | Large model + large image | Switch to `sam2.1_hiera_base_plus.pt` in Cell 2; resize image to max 2000px |
| 0 or very few fragments | Thresholds too strict | Lower `PRED_IOU_THRESH` to 0.80 and `STABILITY_SCORE_THRESH` to 0.88 in Cell 4 |
| Too many fragments (noise) | Thresholds too loose | Raise `MIN_FRAGMENT_AREA_PX` to 1500–2000 in Cell 4 |
| Rod tips not detected | Colour out of HSV range | Try `ROD_COLOR = 'orange'`; or manually set `pixel_to_mm` in Cell 5 |
| Sizes look too large / small | Wrong `ROD_KNOWN_LENGTH_MM` | Measure the rod physically in mm and update Cell 4 |
| Rosin-Rammler fit fails | Too few fragments (<30) | Lower `PRED_IOU_THRESH` and `MIN_FRAGMENT_AREA_PX`; or use `POINTS_PER_SIDE = 64` |